# RFP Parsing P2 250 Pipeline

Goal: build V1/V2 수정버전 outputs for a 250-document pilot set.

- This is not V3.
- Output folder: `outputs/parsing_p2_250`
- Existing `outputs/parsing_v1_v2_2차` is not overwritten.
- V1/V2 chunking is preserved: 1000 chars / 150 overlap.
- Added P2 fixes: unique chunk_id, toc chunk separation, artifact cleaner reinforcement.


## Large File Note

원본 HWP/PDF와 parsing output(JSONL/XLSX/CSV)은 용량이 크기 때문에 GitHub에 포함하지 않습니다. 실행 전 공유 Drive 또는 로컬/VM 디스크에 `data/`와 `outputs/` 구조를 먼저 배치합니다.


In [1]:
# 0. Install requirements for a fresh virtual environment
# This cell is self-contained. Run it once in a new environment, then restart the kernel.
# For automated re-execution in an already prepared environment, set RFP_SKIP_INSTALL=1.

from pathlib import Path
import os
import subprocess
import sys


def find_project_dir_for_install() -> Path:
    candidates = []
    env_project_dir = os.getenv("RFP_PROJECT_DIR")
    if env_project_dir:
        candidates.append(Path(env_project_dir).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    seen = set()
    unique_candidates = []
    for path in candidates:
        if path not in seen:
            seen.add(path)
            unique_candidates.append(path)

    for path in unique_candidates:
        if (path / "requirements.txt").exists():
            return path

    checked = "\n".join(str(path) for path in unique_candidates[:12])
    raise FileNotFoundError(
        "requirements.txt not found. Run this notebook inside project_2nd, "
        "or set RFP_PROJECT_DIR to the project_2nd path.\n\n"
        f"checked:\n{checked}"
    )


INSTALL_PROJECT_DIR = find_project_dir_for_install()
INSTALL_REQUIREMENTS_TXT = INSTALL_PROJECT_DIR / "requirements.txt"

print("PROJECT_DIR:", INSTALL_PROJECT_DIR)
if os.getenv("RFP_SKIP_INSTALL") == "1":
    print("RFP_SKIP_INSTALL=1, requirements installation skipped for this run.")
else:
    cmd = [sys.executable, "-m", "pip", "install", "-r", str(INSTALL_REQUIREMENTS_TXT)]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    print("requirements installed. Restart the kernel, then continue from the next cell.")


PROJECT_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd
RFP_SKIP_INSTALL=1, requirements installation skipped for this run.


In [2]:
# 1. Project path configuration
# Large input files are intentionally not committed. Place data/outputs on Drive, local disk, or VM before running.
# Teammates should only need to adjust the working directory or RFP_PROJECT_DIR.

from pathlib import Path
import os
import sys


def find_project_dir() -> Path:
    candidates = []
    env_project_dir = os.getenv("RFP_PROJECT_DIR")
    if env_project_dir:
        candidates.append(Path(env_project_dir).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    seen = set()
    unique_candidates = []
    for path in candidates:
        if path not in seen:
            seen.add(path)
            unique_candidates.append(path)

    for path in unique_candidates:
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path

    checked = "\n".join(str(path) for path in unique_candidates[:12])
    raise FileNotFoundError(
        "Project root not found. Run this notebook inside project_2nd, "
        "or set RFP_PROJECT_DIR to the project_2nd path.\n\n"
        f"checked:\n{checked}"
    )


PROJECT_DIR = find_project_dir()
PARSING_OUTPUT_NAME = "parsing_p2_250"
PARSING_VERSION_LABEL = "p2_chunkfix_toc_clean"

PARSING_SRC_DIR = PROJECT_DIR / "src" / "parsing"
for import_path in [PARSING_SRC_DIR, PROJECT_DIR]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from rfp_parsing_v1_v2_lib import make_project_paths

PATHS = make_project_paths(
    PROJECT_DIR,
    parsing_output_name=PARSING_OUTPUT_NAME,
    parsing_version_label=PARSING_VERSION_LABEL,
)

print("PROJECT_DIR:", PROJECT_DIR)
print("PARSING_OUTPUT_NAME:", PARSING_OUTPUT_NAME)
print("PARSING_VERSION_LABEL:", PARSING_VERSION_LABEL)
print("OUTPUT_DESCRIPTION:", PATHS["output_description"])
print("DATA_DIR:", PATHS["data_dir"])
print("ORIGINAL_DATA_DIR:", PATHS["original_data_dir"])
print("EVAL_DIR:", PATHS["eval_dir"])
print("PARSING_OUTPUT_DIR:", PATHS["parsing_output_dir"])
print("METADATA_EXCEL:", PATHS["metadata_excel"])
print("Python:", sys.executable)


PROJECT_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd
PARSING_OUTPUT_NAME: parsing_v1_v2_revision
PARSING_VERSION_LABEL: v1_v2_revision
OUTPUT_DESCRIPTION: V1/V2 수정버전
DATA_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\data
ORIGINAL_DATA_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\data\original_data_list
EVAL_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\data\eval
PARSING_OUTPUT_DIR: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision
METADATA_EXCEL: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\rfp_parsing_metadata_250_v1_v2_revision.xlsx
Python: C:\Users\yoosy\Desktop\codeit\project_2nd\.venv\Scripts\python.exe


## Fixed Scope

- Pilot: 250 documents total, including all 62 eval ground-truth documents.
- V1: section/chapter-aware text blocks.
- V2: V1 text blocks plus table-like row/dict blocks and budget/submission candidates.
- Deferred: image OCR, vision embeddings, exact page matching.
- Required summary: document counts, success/failure counts, block counts, and chunk counts.


In [3]:
# 2. Load constants and artifact policy

from rfp_parsing_v1_v2_lib import (
    PILOT_TOTAL_DOCS,
    EVAL_GROUND_TRUTH_DOCS_TOTAL,
    EVAL_PHYSICAL_SOURCE_DOCS_TOTAL,
    ADDITIONAL_SAMPLED_DOCS,
    ARTIFACT_REMOVE_TOKENS,
    CONFIRMED_KEEP_HANJA_TOKENS,
    KEEP_HANJA_RUNS,
    CHUNK_MAX_CHARS,
    CHUNK_OVERLAP,
)

print("pilot docs:", PILOT_TOTAL_DOCS)
print("eval ground-truth aliases:", EVAL_GROUND_TRUTH_DOCS_TOTAL)
print("eval physical source docs:", EVAL_PHYSICAL_SOURCE_DOCS_TOTAL)
print("additional sampled docs:", ADDITIONAL_SAMPLED_DOCS)
print("artifact remove tokens:", sorted(ARTIFACT_REMOVE_TOKENS))
print("confirmed keep hanja tokens:", sorted(CONFIRMED_KEEP_HANJA_TOKENS))
print("keep hanja runs:", sorted(KEEP_HANJA_RUNS))
print("chunk max chars:", CHUNK_MAX_CHARS)
print("chunk overlap:", CHUNK_OVERLAP)
print("retrieval signals: project_name, issuer, final_notice_id, final_budget, final dates, final periods/deadlines, final eligibility terms, exact_terms, dates, amounts, submission_doc_terms")


pilot docs: 250
eval ground-truth aliases: 62
eval physical source docs: 61
additional sampled docs: 189
artifact remove tokens: ['普', '楴', '汫', '沤', '浥', '浫', '浵', '潣', '爔', '牦', '蕀', '遠']
confirmed keep hanja tokens: ['丁', '丙', '乙', '內', '共', '外', '新', '有', '未', '案', '無', '甲', '舊', '過']
keep hanja runs: ['內外', '共有', '有償', '未定', '案內', '無償', '甲乙', '甲乙丙', '甲乙丙丁']
chunk max chars: 1000
chunk overlap: 150
retrieval signals: project_name, issuer, final_notice_id, final_budget, final dates, final periods/deadlines, final eligibility terms, exact_terms, dates, amounts, submission_doc_terms


In [4]:
# 3. Build pilot_docs_250.csv
# Includes the 62 eval documents first, then fills the remaining 188 with budget/submission/evaluation/form-heavy documents.

from rfp_parsing_v1_v2_lib import build_pilot_docs

pilot_docs_df, eval_docs_df, original_inventory_df, metadata_df = build_pilot_docs(PATHS)

print("saved:", PATHS["pilot_docs_csv"])
print("eval ground truth docs:", len(eval_docs_df))
print("original inventory docs:", len(original_inventory_df))
print("metadata rows:", len(metadata_df))
print("pilot_total_docs:", len(pilot_docs_df))
print("eval_aliases_covered:", EVAL_GROUND_TRUTH_DOCS_TOTAL)
print("eval_physical_source_docs_included:", int(pilot_docs_df["is_eval_ground_truth"].sum()))
print("additional_sampled_docs:", int((~pilot_docs_df["is_eval_ground_truth"].astype(bool)).sum()))
print("\nfile_type counts:")
print(pilot_docs_df["file_type"].value_counts().to_string())
print("\nsampling_reason counts:")
print(pilot_docs_df["sampling_reason"].value_counts().head(20).to_string())

display(pilot_docs_df.head(10))


saved: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\pilot_docs_250.csv
eval ground truth docs: 61
original inventory docs: 690
metadata rows: 0
pilot_total_docs: 250
eval_aliases_covered: 62
eval_physical_source_docs_included: 61
additional_sampled_docs: 189

file_type counts:
file_type
hwp    247
pdf      3

sampling_reason counts:
sampling_reason
appendix_forms+budget+eligibility+evaluation_table+submission_docs                 126
eval_ground_truth                                                                   61
appendix_forms+eligibility+evaluation_table+submission_docs                         26
appendix_forms+budget+eligibility+submission_docs                                   23
appendix_forms+budget+evaluation_table+submission_docs                               7
appendix_forms+eligibility+submission_docs                                           2
budget+eligibility+evaluation_table+submission_docs                                  2
appendix_for

In [5]:
# 4. Run V1/V2 parsing pipeline
# Reads the 250 pilot documents and writes parsed_blocks/chunks/summary outputs.

from rfp_parsing_v1_v2_lib import run_parsing_pipeline, create_metadata_excel

summary, doc_parse_summary_df = run_parsing_pipeline(PATHS, pilot_docs_df=pilot_docs_df)
metadata_excel_path = create_metadata_excel(PATHS)

print("saved outputs:")
for key in [
    "pilot_docs_csv",
    "doc_parse_summary_csv",
    "parsed_blocks_v1_jsonl",
    "parsed_blocks_v2_jsonl",
    "chunks_v1_jsonl",
    "chunks_v2_jsonl",
    "parsing_summary_json",
    "parsing_summary_md",
    "metadata_excel",
]:
    print("-", PATHS[key])

print("\nsummary:")
for key, value in summary.items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} items")
    else:
        print(f"{key}: {value}")

display(doc_parse_summary_df.head(10))


saved outputs:
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\pilot_docs_250.csv
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\doc_parse_summary.csv
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\parsed_blocks_v1.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\parsed_blocks_v2.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\chunks_v1.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\chunks_v2.jsonl
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\parsing_summary.json
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\parsing_summary.md
- C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\rfp_parsing_metadata_250_v1_v2_revision.xlsx

summary:
output_description: V1/V2 수정버전
parsing_output_name: parsing_v1_v2_revision
parsing_version_label

parse pilot docs: 100%|##########| 250/250 [01:56<00:00,  2.15it/s]


In [6]:
# 5. Final validation and quick preview

import json
import pandas as pd

summary = json.loads(PATHS["parsing_summary_json"].read_text(encoding="utf-8"))

assert summary["pilot_total_docs"] == PILOT_TOTAL_DOCS
assert summary["eval_docs_included"] == EVAL_GROUND_TRUTH_DOCS_TOTAL
assert summary["eval_physical_source_docs_included"] == EVAL_PHYSICAL_SOURCE_DOCS_TOTAL
assert summary["parse_success_docs"] + summary["parse_failed_docs"] == PILOT_TOTAL_DOCS
assert summary["v2_chunks"] >= summary["v1_chunks"] or summary["v2_chunks"] == 0
assert summary["parsing_output_name"] == PARSING_OUTPUT_NAME
assert summary["parsing_version_label"] == PARSING_VERSION_LABEL

unknown_notice_count = 0
with PATHS["chunks_v2_jsonl"].open("r", encoding="utf-8") as f:
    for line in f:
        if "공고번호: unknown" in line:
            unknown_notice_count += 1
assert unknown_notice_count == 0

print("validation passed")
print("summary path:", PATHS["parsing_summary_json"])
print("markdown summary path:", PATHS["parsing_summary_md"])
print("metadata excel path:", PATHS["metadata_excel"])
print("unknown notice header count:", unknown_notice_count)
for key in [
    "docs_with_final_budget",
    "docs_with_date_candidates",
    "docs_with_final_bid_deadline",
    "docs_with_period_candidates",
    "docs_with_final_project_duration",
    "docs_with_final_maintenance_period",
    "docs_with_final_warranty_period",
    "docs_with_final_deadline_terms",
    "docs_with_eligibility_candidates",
    "docs_with_final_bid_eligibility_terms",
]:
    print(f"{key}:", summary.get(key))

preview_doc_summary = pd.read_csv(PATHS["doc_parse_summary_csv"], encoding="utf-8-sig")
display(preview_doc_summary.head(10))

metadata_preview = pd.read_excel(PATHS["metadata_excel"], sheet_name="metadata_250")
print("\nmetadata columns:")
print(metadata_preview.columns.tolist())
display(metadata_preview[[
    "공고 번호",
    "사업명",
    "사업 금액",
    "사업금액_원",
    "사업기간",
    "무상유지보수기간",
    "하자담보책임기간",
    "기한/기간 기타",
    "입찰참가자격",
    "제출서류",
    "원문텍스트_20000자",
]].head(5))

period_candidates_preview = pd.read_excel(PATHS["metadata_excel"], sheet_name="period_candidates")
eligibility_candidates_preview = pd.read_excel(PATHS["metadata_excel"], sheet_name="eligibility_candidates")
print("\nperiod candidate rows:", len(period_candidates_preview))
print("eligibility candidate rows:", len(eligibility_candidates_preview))
display(period_candidates_preview.head(10))
display(eligibility_candidates_preview.head(10))

with PATHS["chunks_v2_jsonl"].open("r", encoding="utf-8") as f:
    for _ in range(3):
        line = f.readline().strip()
        if not line:
            break
        item = json.loads(line)
        print("\n--- chunk preview ---")
        print("chunk_id:", item["chunk_id"])
        print("source_file:", item["source_file"])
        print("chunk_type:", item["chunk_type"])
        print("issuer:", item.get("issuer"))
        print("project_name:", item.get("project_name"))
        print("final_notice_id:", item.get("final_notice_id"))
        print("final_budget:", item.get("final_budget"))
        print("final_bid_deadline:", item.get("final_bid_deadline"))
        print("final_project_duration:", item.get("final_project_duration"))
        print("final_maintenance_period:", item.get("final_maintenance_period"))
        print("final_warranty_period:", item.get("final_warranty_period"))
        print("final_deadline_terms:", item.get("final_deadline_terms"))
        print("final_bid_eligibility_terms:", str(item.get("final_bid_eligibility_terms", ""))[:200])
        print("exact_terms:", item.get("exact_terms", [])[:8])
        print("amounts:", item.get("amounts", [])[:5])
        print("dates:", item.get("dates", [])[:5])
        print(item["content"][:500])


validation passed
summary path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\parsing_summary.json
markdown summary path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\parsing_summary.md
metadata excel path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\rfp_parsing_metadata_250_v1_v2_revision.xlsx
unknown notice header count: 0
docs_with_final_budget: 210
docs_with_date_candidates: 161
docs_with_final_bid_deadline: 40
docs_with_period_candidates: 250
docs_with_final_project_duration: 241
docs_with_final_maintenance_period: 57
docs_with_final_warranty_period: 197
docs_with_final_deadline_terms: 151
docs_with_eligibility_candidates: 250
docs_with_final_bid_eligibility_terms: 250
  pilot_doc_id          doc_id                                        norm_name                                          source_file file_type  is_eval_ground_truth parser_status                parser  raw_char_len  clean_char_len  e

## Sanity Checks

The cells below inspect the already-created `parsing_p2_250` artifacts.
They do not rebuild parsing outputs.


In [4]:
# 6. Sanity check: artifact existence, workbook sheets, and row counts

import json
import re
from pathlib import Path

import pandas as pd

revision_dir = PATHS["parsing_output_dir"]
excel_path = PATHS["metadata_excel"]

required_artifacts = [
    "pilot_docs_250.csv",
    "doc_parse_summary.csv",
    "parsed_blocks_v1.jsonl",
    "parsed_blocks_v2.jsonl",
    "chunks_v1.jsonl",
    "chunks_v2.jsonl",
    "parsing_summary.json",
    "parsing_summary.md",
    "rfp_parsing_metadata_250_v1_v2_revision.xlsx",
]

artifact_check = pd.DataFrame([
    {
        "file": name,
        "exists": (revision_dir / name).exists(),
        "size_mb": round((revision_dir / name).stat().st_size / 1024 / 1024, 2) if (revision_dir / name).exists() else None,
        "updated_at": (revision_dir / name).stat().st_mtime if (revision_dir / name).exists() else None,
    }
    for name in required_artifacts
])

summary = json.loads(PATHS["parsing_summary_json"].read_text(encoding="utf-8"))
metadata_df = pd.read_excel(excel_path, sheet_name="metadata_250")
doc_summary_df = pd.read_csv(PATHS["doc_parse_summary_csv"], encoding="utf-8-sig")
workbook = pd.ExcelFile(excel_path)

print("revision_dir:", revision_dir)
print("excel_path:", excel_path)
print("output_description:", summary.get("output_description"))
print("parsing_output_name:", summary.get("parsing_output_name"))
print("parsing_version_label:", summary.get("parsing_version_label"))
print("workbook sheets:", workbook.sheet_names)
print("metadata rows:", len(metadata_df))
print("doc summary rows:", len(doc_summary_df))
display(artifact_check)


revision_dir: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision
excel_path: C:\Users\yoosy\Desktop\codeit\project_2nd\outputs\parsing_v1_v2_revision\rfp_parsing_metadata_250_v1_v2_revision.xlsx
output_description: V1/V2 수정버전
parsing_output_name: parsing_v1_v2_revision
parsing_version_label: v1_v2_revision
workbook sheets: ['metadata_250', 'summary', 'doc_parse_summary', 'notice_id_candidates', 'budget_candidates', 'date_candidates', 'period_candidates', 'eligibility_candidates', 'submission_candidates', 'final_submission_docs', 'final_eligibility_terms']
metadata rows: 250
doc summary rows: 250
                                           file  exists  size_mb    updated_at
0                            pilot_docs_250.csv    True     0.13  1.778839e+09
1                         doc_parse_summary.csv    True     0.62  1.778839e+09
2                        parsed_blocks_v1.jsonl    True   187.05  1.778839e+09
3                        parsed_blocks_v2.jsonl    True   3

In [5]:
# 7. Sanity check: key metadata coverage and missing examples

def present(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.strip()

coverage_columns = [
    "공고 번호",
    "사업 금액",
    "사업금액_원",
    "공개 일자",
    "입찰 참여 시작일",
    "입찰 참여 마감일",
    "사업기간",
    "무상유지보수기간",
    "하자담보책임기간",
    "기한/기간 기타",
    "입찰참가자격",
    "제출서류",
]

coverage_rows = []
for column in coverage_columns:
    values = present(metadata_df[column]) if column in metadata_df.columns else pd.Series([""] * len(metadata_df))
    coverage_rows.append({
        "column": column,
        "filled_docs": int((values != "").sum()),
        "missing_docs": int((values == "").sum()),
        "filled_rate": round(float((values != "").mean()), 3),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

print("budget status counts:")
display(metadata_df["사업금액_상태"].fillna("missing").value_counts(dropna=False).rename_axis("status").reset_index(name="docs"))

for label, column in [
    ("사업금액 결측 예시", "사업 금액"),
    ("공개일자 결측 예시", "공개 일자"),
    ("입찰시작일 결측 예시", "입찰 참여 시작일"),
    ("입찰마감일 결측 예시", "입찰 참여 마감일"),
    ("공고번호 결측 예시", "공고 번호"),
]:
    missing = metadata_df[present(metadata_df[column]) == ""]
    print("\n" + label + f": {len(missing)} docs")
    display(missing[["파일명", "사업명", "발주 기관", "원문텍스트_20000자"]].head(5))


       column  filled_docs  missing_docs  filled_rate
0       공고 번호            2           248        0.008
1       사업 금액          210            40        0.840
2      사업금액_원          210            40        0.840
3       공개 일자           46           204        0.184
4   입찰 참여 시작일            7           243        0.028
5   입찰 참여 마감일           40           210        0.160
6        사업기간          241             9        0.964
7    무상유지보수기간           57           193        0.228
8    하자담보책임기간          197            53        0.788
9    기한/기간 기타          151            99        0.604
10     입찰참가자격          250             0        1.000
11       제출서류          237            13        0.948
budget status counts:
           status  docs
0       extracted   210
1         missing    27
2  candidate_only    13

사업금액 결측 예시: 40 docs
                                                  파일명                                     사업명           발주 기관                                                  

In [6]:
# 8. Sanity check: revision-specific schema and anti-regression checks

required_sheets = {
    "metadata_250",
    "summary",
    "doc_parse_summary",
    "notice_id_candidates",
    "budget_candidates",
    "date_candidates",
    "period_candidates",
    "eligibility_candidates",
    "submission_candidates",
    "final_submission_docs",
    "final_eligibility_terms",
}
required_columns = {
    "사업기간",
    "사업기간_근거",
    "무상유지보수기간",
    "무상유지보수기간_근거",
    "하자담보책임기간",
    "하자담보책임기간_근거",
    "기한/기간 기타",
    "기한/기간 기타_근거",
    "입찰참가자격",
    "입찰참가자격_근거",
    "원문텍스트_20000자",
}

unknown_notice_header_count = 0
with PATHS["chunks_v2_jsonl"].open("r", encoding="utf-8") as f:
    for line in f:
        if "공고번호: unknown" in line:
            unknown_notice_header_count += 1

notice_column_count = sum(1 for col in metadata_df.columns if "공고" in str(col) and "번호" in str(col))
schema_check = {
    "is_v1_v2_revision": summary.get("output_description") == "V1/V2 수정버전",
    "is_not_v3_label": "v3" not in str(summary.get("parsing_version_label", "")).lower(),
    "required_sheets_present": required_sheets.issubset(set(workbook.sheet_names)),
    "required_columns_present": required_columns.issubset(set(metadata_df.columns)),
    "notice_column_count_is_1": notice_column_count == 1,
    "unknown_notice_header_count_is_0": unknown_notice_header_count == 0,
    "pilot_docs_250": len(metadata_df) == 250,
    "parse_success_250": int(summary.get("parse_success_docs", 0)) == 250,
}

display(pd.DataFrame([{"check": key, "passed": value} for key, value in schema_check.items()]))
print("notice_column_count:", notice_column_count)
print("unknown_notice_header_count:", unknown_notice_header_count)
assert all(schema_check.values()), "One or more revision sanity checks failed."


                              check  passed
0                 is_v1_v2_revision    True
1                   is_not_v3_label    True
2           required_sheets_present    True
3          required_columns_present    True
4          notice_column_count_is_1    True
5  unknown_notice_header_count_is_0    True
6                    pilot_docs_250    True
7                 parse_success_250    True
notice_column_count: 1
unknown_notice_header_count: 0


In [7]:
# 9. Sanity check: sample documents for revised budget/period/eligibility extraction

period_candidates_df = pd.read_excel(excel_path, sheet_name="period_candidates")
eligibility_candidates_df = pd.read_excel(excel_path, sheet_name="eligibility_candidates")
budget_candidates_df = pd.read_excel(excel_path, sheet_name="budget_candidates")

sample_name_parts = [
    "고려대학교",
    "한국가스공사",
    "한국수자원공사_수도사업장 통합 사고분석솔루션",
]

sample_cols = [
    "파일명",
    "사업 금액",
    "사업금액_원",
    "사업기간",
    "사업기간_근거",
    "무상유지보수기간",
    "무상유지보수기간_근거",
    "하자담보책임기간",
    "하자담보책임기간_근거",
    "기한/기간 기타",
    "기한/기간 기타_근거",
    "입찰참가자격",
]

for name_part in sample_name_parts:
    sample = metadata_df[metadata_df["파일명"].astype(str).str.contains(name_part, regex=False, na=False)]
    print("\n=== sample:", name_part, "===")
    display(sample[sample_cols].head(3))
    if sample.empty:
        continue
    source_file = sample.iloc[0]["파일명"]
    print("period candidates:")
    display(
        period_candidates_df[period_candidates_df["source_file"].astype(str) == str(source_file)]
        [["period_type", "period_value", "score", "is_form_context", "raw_text"]]
        .head(10)
    )
    print("budget candidates:")
    display(
        budget_candidates_df[budget_candidates_df["source_file"].astype(str) == str(source_file)]
        [["raw_amount", "amount_krw", "score", "context"]]
        .head(8)
    )
    print("eligibility candidates:")
    display(
        eligibility_candidates_df[eligibility_candidates_df["source_file"].astype(str) == str(source_file)]
        [["matched_terms", "score", "is_form_context", "raw_text"]]
        .head(8)
    )



=== sample: 고려대학교 ===
                              파일명            사업 금액        사업금액_원            사업기간                                                                                  사업기간_근거       무상유지보수기간                                                                                         무상유지보수기간_근거 하자담보책임기간                                                                                                                         하자담보책임기간_근거 기한/기간 기타 기한/기간 기타_근거                                                                                    입찰참가자격
0  고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf  11,270,000,000원  1.127000e+10  계약일로부터 24개월 이내  가. 사업명: 고려대학교 차세대 포털·학사 정보시스템 구축 사업 나. 사업기간: 계약일로부터 24개월 이내 다. 무상유지보수기간 : 사업종료일로부터 12개월  사업종료일로부터 12개월  나. 사업기간: 계약일로부터 24개월 이내 다. 무상유지보수기간 : 사업종료일로부터 12개월 라. 사업예산 : 11,270,000,000원 (V.A.T 포함, 3년 분할 지급)     12개월  보장하여야 하며, 우리 학교에서 운영 중인 각종 S/W와 연동이 되어야 함 ❍ 무상 하자보수 기간 시작일은 최종 검수일로 하고, S/W 기본 12개월의 무상 하자보수 기간을 적용하되 추가 무상 하자보수 기간은 제안사 자율로 제안할 수      NaN      

In [8]:
# 10. Sanity check: suspicious final values and candidate/final separation

period_final_columns = [
    "사업기간",
    "무상유지보수기간",
    "하자담보책임기간",
    "기한/기간 기타",
]

suspicious_patterns = [
    r"\b0+\d+\s*(일|개월|년)",
    r"\b\d{1,2}\s*일\b",
    r"제\s*\d+\s*(조|항|호)",
]

suspicious_rows = []
for column in period_final_columns:
    values = present(metadata_df[column])
    for pattern in suspicious_patterns:
        hits = metadata_df[values.str.contains(pattern, regex=True, na=False)]
        for _, row in hits.head(20).iterrows():
            suspicious_rows.append({
                "column": column,
                "pattern": pattern,
                "파일명": row.get("파일명", ""),
                "value": row.get(column, ""),
            })

suspicious_df = pd.DataFrame(suspicious_rows)
print("suspicious final period/deadline values:", len(suspicious_df))
display(suspicious_df.head(30))

form_context_candidates = period_candidates_df[period_candidates_df["is_form_context"].astype(str).str.lower().isin(["true", "1"])]
print("period candidates kept only as form/template context:", len(form_context_candidates))
display(form_context_candidates[["source_file", "period_type", "period_value", "score", "raw_text"]].head(10))

external_deadline_candidates = period_candidates_df[
    period_candidates_df["period_value"].fillna("").astype(str).str.contains("공고|참조|따름", regex=True)
]
print("external-reference deadline candidates:", len(external_deadline_candidates))
display(external_deadline_candidates[["source_file", "period_type", "period_value", "score", "raw_text"]].head(10))


suspicious final period/deadline values: 0
Empty DataFrame
Columns: []
Index: []
period candidates kept only as form/template context: 152
                                           source_file       period_type  period_value  score                                                                                                                                                                                                                                                               raw_text
109            한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp  project_duration            1년     42                                                                                                                                                                                                                                       <붙임 1> 소프트웨어 개발사업의 적정 사업기간 종합산정서
258            울산광역시_2024년 버스정보시스템 확대 구축 및 기능개선 용역.hwp  project_duration       12개월 이내     -4                                                      

C:\Users\yoosy\Desktop\codeit\project_2nd\notebooks\parsing\rfp_parsing_v1_v2_pipeline.ipynb#cell-13:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
C:\Users\yoosy\Desktop\codeit\project_2nd\notebooks\parsing\rfp_parsing_v1_v2_pipeline.ipynb#cell-13:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
C:\Users\yoosy\Desktop\codeit\project_2nd\notebooks\parsing\rfp_parsing_v1_v2_pipeline.ipynb#cell-13:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
C:\Users\yoosy\Desktop\codeit\project_2nd\notebooks\parsing\rfp_parsing_v1_v2_pipeline.ipynb#cell-13:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
C:\Users\yoosy\Desktop\codeit\project_2nd\notebo

In [9]:
# 11. Sanity check: compact final report for team sharing

report_rows = [
    ("전체 문서 수", len(metadata_df)),
    ("파싱 성공 문서 수", summary.get("parse_success_docs")),
    ("공고번호 final 존재", int((present(metadata_df["공고 번호"]) != "").sum())),
    ("사업금액 final 존재", int((present(metadata_df["사업 금액"]) != "").sum())),
    ("공개일자 final 존재", int((present(metadata_df["공개 일자"]) != "").sum())),
    ("입찰시작일 final 존재", int((present(metadata_df["입찰 참여 시작일"]) != "").sum())),
    ("입찰마감일 final 존재", int((present(metadata_df["입찰 참여 마감일"]) != "").sum())),
    ("사업기간 final 존재", int((present(metadata_df["사업기간"]) != "").sum())),
    ("무상유지보수기간 final 존재", int((present(metadata_df["무상유지보수기간"]) != "").sum())),
    ("하자담보책임기간 final 존재", int((present(metadata_df["하자담보책임기간"]) != "").sum())),
    ("기한/기간 기타 final 존재", int((present(metadata_df["기한/기간 기타"]) != "").sum())),
    ("입찰참가자격 final 존재", int((present(metadata_df["입찰참가자격"]) != "").sum())),
    ("제출서류 final 존재", int((present(metadata_df["제출서류"]) != "").sum())),
    ("period candidate rows", len(period_candidates_df)),
    ("eligibility candidate rows", len(eligibility_candidates_df)),
    ("unknown 공고번호 header", unknown_notice_header_count),
]

final_report_df = pd.DataFrame(report_rows, columns=["metric", "value"])
display(final_report_df)

print("해석 메모:")
print("- 사업금액/일자/기간은 원문에서 확실하게 찾은 값만 final로 올립니다.")
print("- 공고문 참조/입찰 공고에 따름은 날짜로 만들지 않고 기한/기간 기타 또는 candidate 근거로 둡니다.")
print("- 가격제안서/하도급계획서/별지/서식 안의 작성 공란은 final 승격을 억제합니다.")
print("- candidate sheet는 넓게 잡은 검수 후보이고, metadata_250의 final 컬럼은 더 보수적으로 선별한 값입니다.")


                        metric  value
0                      전체 문서 수    250
1                   파싱 성공 문서 수    250
2                공고번호 final 존재      2
3                사업금액 final 존재    210
4                공개일자 final 존재     46
5               입찰시작일 final 존재      7
6               입찰마감일 final 존재     40
7                사업기간 final 존재    241
8            무상유지보수기간 final 존재     57
9            하자담보책임기간 final 존재    197
10           기한/기간 기타 final 존재    151
11             입찰참가자격 final 존재    250
12               제출서류 final 존재    237
13       period candidate rows   8855
14  eligibility candidate rows  21907
15         unknown 공고번호 header      0
해석 메모:
- 사업금액/일자/기간은 원문에서 확실하게 찾은 값만 final로 올립니다.
- 공고문 참조/입찰 공고에 따름은 날짜로 만들지 않고 기한/기간 기타 또는 candidate 근거로 둡니다.
- 가격제안서/하도급계획서/별지/서식 안의 작성 공란은 final 승격을 억제합니다.
- candidate sheet는 넓게 잡은 검수 후보이고, metadata_250의 final 컬럼은 더 보수적으로 선별한 값입니다.
